# Model predictive control

**Plan the next 2 s, every tick.**

Unit 9's agent drives a racing line, follows it with pure pursuit, and plans its speed with `MU_FRACTION = 0.54` of the measured grip.
Its family edge was 0.56, so 0.54 is one grid step inside it, by the margin rule.
What happens one step **above** that edge, at 0.58?
On `test-circuit-seed4`, a training track, it crashes.

Pure pursuit looks at **one point** ahead and steers for it.
It does not know that the line will turn harder in a second, that its own command acts one tick late (measured lag 1 tick, Unit 8), or that it is already drifting off the line.
What if the steering planned the whole next 2 s instead, every tick?

Here is Unit 9's agent at 0.58 next to this unit's, `agents/unit-10.py`, which steers by **model predictive control** (MPC) at the same 0.58, on `test-circuit-seed4` (`carla_like(noise=True)`, seed 0):

In [ ]:
import math
import time
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar, carla_like
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear, minimize

AGENTS = Path("../../projects/racekit/agents")
UNIT_09, UNIT_10 = AGENTS / "unit-09.py", AGENTS / "unit-10.py"
workdir = Path(mkdtemp(prefix="roar-unit-10-"))
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
DT = 0.05  # s: one control tick
WHEELBASE = 2.875  # m, L (Unit 3)
MAX_STEER = math.radians(66.0)  # rad: the steering gain, measured (Unit 8: 65.7 deg, rounded to 66 as the agents do)
U_MAX = math.tan(MAX_STEER) / WHEELBASE  # 1/m: the tightest curvature full lock drives
NO_TIMING = {"MPC_BUDGET_MS": None}  # every pinned MPC race: no wall clock inside the race (section 7 says why)

# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
# agents/unit-09.py and unit-10.py ship test-circuit's; another track's values are written in with write_variant.
MEASURED_MU_G = {
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
}


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track="test-circuit", seed=0, car=None, max_seconds=450.0):
    # Every race of this lesson is one seed; the car is the carla-like car with its noise unless one is given.
    return run_race(source, track, car=carla_like(noise=True) if car is None else car, seed=seed,
                    max_seconds=max_seconds)


# Unit 9's line solver, as Unit 9 built it: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# Unit 9's line-relative clean rule.
def line_error(result, offsets):
    # The largest |cross-track error from the race's own line| over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def summary(result, offsets):
    verdict = "clean" if clean(result, offsets) else "NOT clean"
    finished = "finished" if result.finished else "did not finish"
    return (f"{verdict}: {result.total_time:.2f} s, {result.collisions} collisions, {finished}, "
            f"{line_error(result, offsets):.2f} m from its line")


def on_track(source, track_name, control, name, **constants):
    # A submission with this track's line and measured grip written in (Unit 9's idiom), plus any constants.
    return variant(source, name, LINE_CONTROL=tuple(float(value) for value in control),
                   MU_G=MEASURED_MU_G[track_name], **constants)


seed4 = load_track("test-circuit-seed4")
seed4_control = solve_line(seed4)
seed4_offsets = spline_basis(len(seed4)) @ seed4_control
pursuit_058 = race(on_track(UNIT_09, "test-circuit-seed4", seed4_control, "u09-seed4-0.58", MU_FRACTION=0.58),
                   track="test-circuit-seed4")
mpc_058 = race(on_track(UNIT_10, "test-circuit-seed4", seed4_control, "u10-seed4", **NO_TIMING),
               track="test-circuit-seed4")
print("Unit 9, pure pursuit, f = 0.58:", summary(pursuit_058, seed4_offsets))
print("Unit 10, MPC,          f = 0.58:", summary(mpc_058, seed4_offsets))
assert not clean(pursuit_058, seed4_offsets)  # U10.3
assert clean(mpc_058, seed4_offsets) and abs(mpc_058.total_time - 271.10) <= TICK  # U10.3

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, result in (("Unit 9: pure pursuit", pursuit_058), ("Unit 10: MPC", mpc_058)):
    telemetry = result.telemetry
    target = seed4_offsets[(telemetry["progress"] + 1) % len(seed4_offsets)]
    error = np.where(telemetry["held"] == 0, telemetry["offset"] - target, np.nan)  # blank while held after a crash
    first = telemetry["t"] < 150.0
    ax.plot(telemetry["t"][first], error[first], label=name)
ax.axhline(CLEAN_ERROR, color="grey", linestyle="--")
ax.axhline(-CLEAN_ERROR, color="grey", linestyle="--")
ax.set_xlabel("t (s)")
ax.set_ylabel("error from the line (m, left +)")
ax.set_title("test-circuit-seed4, f = 0.58, seed 0: the first 150 s")
ax.legend()
plt.show()

Pure pursuit at 0.58 spins and hits the wall again and again and never finishes; MPC, with **the same line, the same speed plan and the same fraction**, is clean in 271.10 s, never more than 1.34 m from its line.
The plot shows the first crash: at about 75 s both cars swing towards the 2 m mark in the same corner; MPC turns back at 1.4 m, while pursuit swings from side to side, wider each time, and crashes at about 84 s (the gap after it is the respawn hold).
Over seeds 0–4 (plan `roar-009`'s routed gate U10.3, re-raced by the milestone) pursuit is not clean on any seed and MPC is clean on all five, median 271.10 s.

Only the steering changed.
This unit builds that steering: a **model** of how the car's error from the line evolves (§1–§2), a plan for the next 40 ticks found by **one least-squares solve** (§3), checked against an exact solver (§4), tuned on the right car (§5), and measured honestly against pure pursuit (§6).
Then the price: a solve every tick must fit in the tick (§7).

## 1. The error model, in the path frame

MPC needs a model that predicts what a steering command will do.
Unit 3's kinematic bicycle predicts (x, y, ψ, v), but for steering only two numbers matter, both measured on the agent's own line points (Unit 9):

- the **cross-track error** e, the car's signed distance from the line's segment (left positive), and
- the **heading error** ψ, the car's yaw minus the segment's heading.

Choose the **path curvature** u that the car drives as the input; the wheel angle that drives it is δ = atan(L u) (Unit 5's arc).
If the line has curvature κ at the car, then while the car moves at speed v,

$$\dot e = v \sin\psi, \qquad \dot\psi = v\left(u - \frac{\kappa\cos\psi}{1 - \kappa e}\right).$$

The second term is how fast the line itself turns under the car: a car that drives exactly the line's curvature (u = κ) on the line (e = ψ = 0) stays there.

**Linearise about the line.**
On the line, e and ψ are small, so sin ψ ≈ ψ, cos ψ ≈ 1 and κe ≈ 0:

$$\dot e = v\,\psi, \qquad \dot\psi = v\,(u - \kappa).$$

**Discrete time.**
The agent acts once a tick, so take one Euler step of Δt = 0.05 s, and write a_t = Δt·v_t:

$$e_{t+1} = e_t + a_t\,\psi_t, \qquad \psi_{t+1} = \psi_t + a_t\,(u_t - \kappa_t).$$

That is **linear** in (e, ψ, u), which is what will make the plan one least-squares solve.
Both models are kinematic: they say nothing about tyres that slip.
The carla-like car is not kinematic: its tyres slip (Unit 6). So this is a model, not the truth; §5 shows what that costs.

How good is the linearisation?
Take the tightest corner of `test-circuit`'s line (radius about 15 m), at 18 m/s, and step both models from the same start with the command u = κ, the line's own curvature: once (one tick), and then 40 times (2 s), with no correction:

In [ ]:
def linear_step(e, psi, u, a, kappa):
    # One tick of the linearised error model (a = DT * v).
    return e + a * psi, psi + a * (u - kappa)


def nonlinear_step(e, psi, u, a, kappa):
    # One tick of the path-frame model before linearising (the same Euler tick).
    return e + a * math.sin(psi), psi + a * (u - kappa * math.cos(psi) / (1.0 - kappa * e))


with shim_installed():
    Unit10, _ = load_submission(UNIT_10)
track = load_track("test-circuit")
agent = Unit10(track.waypoints, None)  # the shipped file on test-circuit: its line, its profile
profile, curvature = np.array(agent.profile), np.array(agent.curvature)
segment_length = np.array(agent.segment_length)
line_offsets = np.array(agent.offsets)
corner_kappa = curvature[int(np.argmax(np.abs(curvature)))]
print(f"the tightest point of the line: curvature {corner_kappa:.4f} 1/m, radius {1 / abs(corner_kappa):.1f} m")

for e0, psi0 in ((0.8, 0.05), (0.8, 0.5)):
    lin, non = (e0, psi0), (e0, psi0)
    for t in range(40):
        lin = linear_step(*lin, corner_kappa, DT * 18.0, corner_kappa)
        non = nonlinear_step(*non, corner_kappa, DT * 18.0, corner_kappa)
        if t == 0:
            print(f"start e = {e0} m, psi = {psi0} rad: after 1 tick the models differ by "
                  f"{abs(lin[0] - non[0]):.4f} m in e and {abs(lin[1] - non[1]):.4f} rad in psi")
    print(f"{'':30}after 2 s the linear model says e = {lin[0]:5.2f} m, the nonlinear one {non[0]:5.2f} m")

One tick of either model is almost the same: from ψ = 0.05 rad the two differ by 0.003 rad in ψ (mostly the κe term: 0.8 m off a 15 m radius), and from ψ = 0.5 rad by 0.02 m in e.
Held blind for 2 s, the small differences add up: the linear model puts the car 2.60 m left of the line, the nonlinear one 0.13 m right of it; and from a heading error of 0.5 rad (about 29°) the linear model is useless (18.80 m against 1.19 m).

Two things follow.
The linear model is good **near the line, for a short time**.
And the agent never holds a command blind for 2 s: it plans again every tick from the measured errors (§3), so the model only has to be right about what the next commands will do, near the line.
§4 measures what the linearisation costs on a real plan.

## 2. A model that changes along the horizon, and the lag

**Time-varying.**
Over the next 2 s the car does not keep its speed, and the line does not keep its curvature.
The agent reads both off its own plan: v_t is Unit 7's profile speed at the distance the car will have travelled after t + 1 ticks, and κ_t is the line's curvature there.
So a_t and κ_t change from tick to tick: the model is **linear time-varying** (LTV).

The profile speed is a **heuristic** reference, not a prediction.
The speed loop (Unit 7's PI) follows the profile, so the agent assumes it will; where the profile speeds up faster than full throttle can (6 m/s², Unit 3's model), it caps the reference at the speed the car can reach; where the profile brakes, it takes the braking as given.
The reference also starts at the start of the car's segment, ignoring where the car is inside it (under 2 m).
These are choices, and they are the agent's: here is its rule, applied before the tightest corner of `test-circuit`:

In [ ]:
A_THROTTLE = 6.0  # m/s^2 at full throttle (Unit 3's model): the fastest the reference may speed up


def horizon_reference(i, speed, n):
    # The agent's rule: the profile speed (capped by full throttle) and the line's curvature at the distance
    # the car will have travelled after t + 1 ticks, t = 0 .. n - 1, starting from segment i.
    count = len(profile)
    v_ref, k_ref = np.empty(n), np.empty(n)
    s_need, k, travelled = 0.0, i, 0.0
    v = max(speed, 1.0)
    for t in range(n):
        s_need += v * DT
        while travelled < s_need:
            travelled += segment_length[k % count]
            k += 1
        v_ref[t] = max(min(profile[k % count], speed + A_THROTTLE * DT * (t + 1)), 1.0)
        k_ref[t] = curvature[k % count]
        v = v_ref[t]
    return v_ref, k_ref


HORIZON = 40  # ticks: 2 s
corner = int(np.argmax(np.abs(curvature)))
start = corner - 20  # about 38 m before the tightest point, braking into it
v_ref, k_ref = horizon_reference(start, profile[start], HORIZON)
print(f"over the horizon: v from {v_ref.min():.1f} to {v_ref.max():.1f} m/s, "
      f"so a_t = DT * v_t from {DT * v_ref.min():.2f} to {DT * v_ref.max():.2f} m; "
      f"kappa from {k_ref.min():.4f} to {k_ref.max():.4f} 1/m")

fig, (ax_v, ax_k) = plt.subplots(1, 2, figsize=(11, 3.2))
ax_v.plot(np.arange(1, HORIZON + 1), v_ref)
ax_v.set_xlabel("tick t + 1")
ax_v.set_ylabel("v_t (m/s)")
ax_v.set_title("the reference speed")
ax_k.plot(np.arange(1, HORIZON + 1), k_ref)
ax_k.set_xlabel("tick t + 1")
ax_k.set_ylabel("kappa_t (1/m)")
ax_k.set_title("the line's curvature")
fig.tight_layout()
plt.show()

Braking into the corner, the reference speed falls from 16.0 to 9.3 m/s over the 2 s, so a_t shrinks from 0.80 to 0.46 m a tick, while the curvature grows from 0.007 to 0.053 1/m.
No single (a, κ) describes the horizon; the steps in both plots are the waypoints, about 2 m apart.

**The lag is part of the model.**
Unit 5 showed that the car applies each command one tick late, and Unit 8 measured it from the residual: lag 1 tick.
So when the agent plans at tick 0, this tick's command will not act until tick 1; during tick 0 the car is still obeying **last tick's** command, which is already queued.
The agent keeps that queue (`MPC_LAG = 1` command long) and first rolls its model through it:

$$(e, \psi) \;\xrightarrow{\;u_{\text{queued}},\ a_0,\ \kappa_0\;}\; (e_1, \psi_1),$$

and then plans the rest of the horizon from (e_1, ψ_1).
The first command of that plan is what the agent sends now; it acts at tick 1, exactly when the plan said it would.
§5 races the agent with `MPC_LAG = 0`, a model that forgets the queue.

## 3. The horizon cost, stacked into one least-squares problem

Over a horizon of m ticks the plan is m commands u_0 … u_{m−1}.
A good plan keeps the errors small and does not jerk the wheel:

$$J(u) = \sum_{t=1}^{m}\left(q_e\,e_t^2 + q_\psi\,\psi_t^2\right) + r\sum_{t=0}^{m-1}\left(u_t - u_{t-1}\right)^2,$$

with u_{−1} the last command the agent applied.
The weights q_e, q_ψ and r are knobs: q_e and q_ψ pay for being off the line, r pays for changing the command.

**Stack the horizon.**
Every e_t and ψ_t is a linear function of the commands, so all 2m of them together are

$$x = G\,u + h, \qquad x = (e_1, \psi_1, e_2, \psi_2, \ldots, e_m, \psi_m),$$

where G (2m × m) says how each command moves each later error, and h is where the errors go with u = 0.
The direct way to build G and h is one tick at a time, carrying the effect of every command so far:

In [ ]:
def loop_build(e, psi, a, kappa):
    # G and h one tick at a time: state' = A_t state + b_t u_t + c_t, A_t = [[1, a_t], [0, 1]], b_t = (0, a_t).
    m = len(a)
    G = np.zeros((2 * m, m))
    h = np.zeros(2 * m)
    state = np.array([e, psi])
    effect = np.zeros((2, m))  # column j: what u_j has done to the current state so far
    for t in range(m):
        A = np.array([[1.0, a[t]], [0.0, 1.0]])
        state = A @ state + np.array([0.0, -a[t] * kappa[t]])
        effect = A @ effect
        effect[:, t] = (0.0, a[t])
        G[2 * t:2 * t + 2] = effect
        h[2 * t:2 * t + 2] = state
    return G, h

That is m small matrix products per tick, 40 of them, every tick of the race.
But the model is simple enough to write G down directly.
A command u_j turns the heading by a_j and the heading stays turned, so for every later tick t ≥ j

$$\frac{\partial \psi_{t+1}}{\partial u_j} = a_j .$$

And that extra heading then moves e by a_{j+1}, a_{j+2}, … a_t, one tick after another; with S_t = a_0 + … + a_t (a running sum, `np.cumsum`),

$$\frac{\partial e_{t+1}}{\partial u_j} = a_j\,(S_t - S_j) .$$

Both are zero for j > t: a command cannot change the past, so both blocks are **lower triangular** (`np.tril`).
h is the same model with u = 0: ψ is pulled by −a_j κ_j, and e collects S_t ψ_0 from the start heading.

In [ ]:
def stack_horizon(e, psi, a, kappa):
    # The closed form: x = G u + h, rows (e_1, psi_1, e_2, psi_2, ...).
    m = len(a)
    S = np.cumsum(a)
    G_psi = np.tril(np.tile(a, (m, 1)))  # row t, column j: a_j for j <= t
    G_e = np.tril(S[:, None] - S[None, :]) * a[None, :]  # a_j (S_t - S_j) for j <= t
    h_psi = psi - G_psi @ kappa
    h_e = e + S * psi - G_e @ kappa
    G = np.empty((2 * m, m))
    G[0::2], G[1::2] = G_e, G_psi
    h = np.empty(2 * m)
    h[0::2], h[1::2] = h_e, h_psi
    return G, h


a_ref = DT * v_ref
G_loop, h_loop = loop_build(0.8, 0.05, a_ref, k_ref)
G, h = stack_horizon(0.8, 0.05, a_ref, k_ref)
print("closed form = tick by tick:", np.allclose(G, G_loop, rtol=1e-12, atol=1e-12),
      np.allclose(h, h_loop, rtol=1e-12, atol=1e-12))
assert np.allclose(G, G_loop, rtol=1e-12, atol=1e-12) and np.allclose(h, h_loop, rtol=1e-12, atol=1e-12)  # U10.1

for build in (loop_build, stack_horizon):
    started = time.perf_counter()
    for _ in range(200):
        build(0.8, 0.05, a_ref, k_ref)
    print(f"{build.__name__}: {(time.perf_counter() - started) / 200 * 1000:.3f} ms per build")

The two builds agree to 1e-12, and the closed form is several times faster: the probe of plan `roar-009` raced both and got the same race to the tick (276.85 s on `test-circuit-seed2026`), with the median step falling from 1.31 to 0.90 ms.

**One least-squares problem.**
Each term of J is a square, so J is the squared length of one long vector, Unit 8's ‖A c − b‖²:

- the error rows: √q_e·e_t and √q_ψ·ψ_t, that is, W (G u + h) with W the diagonal of the weights (√q_e, √q_ψ, √q_e, …, `np.tile`);
- the change rows: √r·(u_t − u_{t−1}), that is, √r (D u − d), with D = I − (the identity shifted one row down; `np.eye(m) - np.eye(m, k=-1)`) and d = (u_{−1}, 0, …, 0).

So the plan is the least-squares solution of

$$\begin{bmatrix} W G \\ \sqrt{r}\,D \end{bmatrix} u \approx \begin{bmatrix} -W h \\ \sqrt{r}\,d \end{bmatrix},$$

one `np.linalg.lstsq` call: no iterations, no tuning of a solver, the exact minimum of J for the linear model.

In [ ]:
WEIGHTS = (0.3, 20.0, 1000.0)  # (q_e, q_psi, r): agents/unit-10.py's MPC_Q_E, MPC_Q_PSI, MPC_R_DU (section 5)


def least_squares_plan(G, h, u_last, weights=WEIGHTS):
    q_e, q_psi, r = weights
    m = G.shape[1]
    w = np.tile([math.sqrt(q_e), math.sqrt(q_psi)], m)
    D = np.eye(m) - np.eye(m, k=-1)
    d = np.zeros(m)
    d[0] = u_last
    rows = np.vstack([w[:, None] * G, math.sqrt(r) * D])
    rhs = np.concatenate([-w * h, math.sqrt(r) * d])
    return np.linalg.lstsq(rows, rhs, rcond=None)[0]


def horizon_cost(e, psi, u, a, kappa, u_last, weights=WEIGHTS, step=linear_step):
    # J, by rolling a model forward tick by tick (no matrices).
    q_e, q_psi, r = weights
    total, previous = 0.0, u_last
    for t in range(len(u)):
        e, psi = step(e, psi, u[t], a[t], kappa[t])
        total += q_e * e * e + q_psi * psi * psi + r * (u[t] - previous) ** 2
        previous = u[t]
    return total


plan = least_squares_plan(G, h, 0.0)
best = horizon_cost(0.8, 0.05, plan, a_ref, k_ref, 0.0)
nudged = min(horizon_cost(0.8, 0.05, plan + 1e-4 * np.sin(k * np.arange(HORIZON)), a_ref, k_ref, 0.0)
             for k in range(1, 6))
print(f"J(plan) = {best:.5f}; the best of five nudged plans: {nudged:.5f}")
assert best < nudged

x = G @ plan + h
fig, (ax_e, ax_u) = plt.subplots(1, 2, figsize=(11, 3.2))
ax_e.plot(np.arange(1, HORIZON + 1), x[0::2], label="e_t (m)")
ax_e.plot(np.arange(1, HORIZON + 1), 10 * x[1::2], label="10 psi_t (rad)")
ax_e.set_xlabel("tick")
ax_e.set_title("the planned errors")
ax_e.legend()
ax_u.plot(np.arange(HORIZON), plan, label="the plan u_t")
ax_u.plot(np.arange(HORIZON), k_ref, label="the line's kappa_t", linestyle="--")
ax_u.set_xlabel("tick")
ax_u.set_ylabel("curvature (1/m)")
ax_u.set_title("the planned commands")
ax_u.legend()
fig.tight_layout()
plt.show()

The plan beats every nudged plan, as the minimum of J must.
Read it: the car starts 0.8 m left of the line, so the plan first steers **right** of the line's curvature (negative u) to point the car back towards the line, then joins the line's curvature as the corner tightens.
The planned e falls from 0.8 m to about 0.1 m over the 2 s; the weights decide how fast (§5).

**Clip, apply, recede.**
The wheel has a full lock, so the first command is clipped to ±tan(δ_max)/L before it is sent as δ = atan(L u).
(Clipping only the first command is the cheapest way to respect the box; a bounded solver such as `lsq_linear` could respect it over the whole horizon, at a cost every tick.)
Then the agent throws the rest of the plan away, and on the next tick, from the new measured errors, plans again: a **receding horizon**.
Most of every plan is never used; it is there so that the first command knows what is coming.

Here is the whole loop on a toy road, 30 m straight, then a 60 m arc of radius 40 m, then straight, at a steady 18 m/s.
The "car" is the nonlinear path-frame model of §1 with a one-tick lag; it starts 0.8 m off the line.
Two controllers: one whose model knows the queued command, one that forgets it:

In [ ]:
def toy_kappa(distance):
    # The toy road: straight, a 60 m arc of radius 40 m from 30 m, straight again.
    return 1.0 / 40.0 if 30.0 <= distance < 90.0 else 0.0


def mpc_tick(e, psi, a, kappa, queue, u_last, weights=WEIGHTS):
    # One MPC tick: roll through the queued commands, stack the rest of the horizon, solve, clip the first.
    for t in range(len(queue)):
        e, psi = linear_step(e, psi, queue[t], a[t], kappa[t])
    G, h = stack_horizon(e, psi, a[len(queue):], kappa[len(queue):])
    plan = least_squares_plan(G, h, u_last, weights)
    return min(max(plan[0], -U_MAX), U_MAX)


def toy_drive(knows_lag, ticks=140, speed=18.0):
    e, psi, distance = 0.8, 0.0, 0.0
    acting = 0.0  # the command the car obeys this tick (sent last tick)
    errors = []
    for _ in range(ticks):
        kappa = np.array([toy_kappa(distance + speed * DT * (t + 1)) for t in range(HORIZON)])
        a = np.zeros(HORIZON) + DT * speed
        queue = [acting] if knows_lag else []
        sent = mpc_tick(e, psi, a, kappa, queue, acting)
        e, psi = nonlinear_step(e, psi, acting, DT * speed, toy_kappa(distance + speed * DT))
        distance += speed * DT
        acting = sent
        errors.append(e)
    return np.array(errors)


with_lag, without_lag = toy_drive(True), toy_drive(False)
for name, errors in (("knows the lag", with_lag), ("forgets the lag", without_lag)):
    print(f"{name:16}: max |e| after the first second {np.max(np.abs(errors[20:])):.3f} m")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(DT * np.arange(1, len(with_lag) + 1), with_lag, label="the model knows the queued command")
ax.plot(DT * np.arange(1, len(without_lag) + 1), without_lag, label="the model forgets it")
ax.axvline(30.0 / 18.0, color="grey", linewidth=0.8)
ax.axvline(90.0 / 18.0, color="grey", linewidth=0.8)
ax.set_xlabel("t (s); between the grey lines: the arc")
ax.set_ylabel("e (m)")
ax.set_title("the receding horizon on a toy road, one-tick lag")
ax.legend()
plt.show()

Both controllers bring the car from 0.8 m onto the line in about 1.5 s and hold it within a few centimetres through the arc: the receding horizon works.
On this toy, the controller that **forgets** the lag does about as well (at worst 0.12 m after the first second, against 0.13 m).
But the toy is kinematic, at a steady speed, measured perfectly: it cannot show what the lag costs on a car that slides.
§5 asks the carla-like car, and its answer is different.

## 4. The exact reference: `scipy.optimize.minimize` on one corner

The least-squares plan is exact for the **linear** model.
How much does the linearisation cost?
On one corner, solve the problem without it: minimise J with the **nonlinear** path-frame model of §1 and the full-lock bounds on every command, using `scipy.optimize.minimize` (method `SLSQP`, which handles bounds).
The corner is the tightest of `test-circuit`'s line: start 20 waypoints before it at a steady 18 m/s, 0.8 m off the line and 0.05 rad off its heading, with the previous command 0, and read κ_t off the line at the distance travelled (straight-line interpolation between waypoints).

In [ ]:
def curvature_at(distances, s, kappa):
    # The line's curvature at each distance along it, by straight-line interpolation between points
    # (s[k]: the distance to point k, increasing).
    out = np.zeros(len(distances))
    k = 0
    for t in range(len(distances)):
        while s[k + 1] < distances[t]:
            k += 1
        share = (distances[t] - s[k]) / (s[k + 1] - s[k])
        out[t] = (1.0 - share) * kappa[k] + share * kappa[k + 1]
    return out


first = corner - 20
s = np.concatenate([[0.0], np.cumsum(segment_length[first:first + 200])])
kappa_c = curvature_at(18.0 * DT * np.arange(1, HORIZON + 1), s[:-1], curvature[first:first + 200])
a_c = np.zeros(HORIZON) + DT * 18.0


def nonlinear_cost(u):
    return horizon_cost(0.8, 0.05, u, a_c, kappa_c, 0.0, step=nonlinear_step)


started = time.perf_counter()
G_c, h_c = stack_horizon(0.8, 0.05, a_c, kappa_c)
u_ls = least_squares_plan(G_c, h_c, 0.0)
ls_ms = (time.perf_counter() - started) * 1000
started = time.perf_counter()
exact = minimize(nonlinear_cost, np.zeros(HORIZON), method="SLSQP", bounds=[(-U_MAX, U_MAX)] * HORIZON,
                 options={"maxiter": 500, "ftol": 1e-12})
scipy_ms = (time.perf_counter() - started) * 1000
above = nonlinear_cost(u_ls) / exact.fun - 1
print(f"scipy (nonlinear model, bounds): J = {exact.fun:.5f} in {scipy_ms:.0f} ms")
print(f"least squares (linear model), J on the nonlinear model = {nonlinear_cost(u_ls):.5f} in {ls_ms:.2f} ms: "
      f"{above:.2%} above")
print(f"the largest difference in the commands: {np.max(np.abs(u_ls - exact.x)):.1e} 1/m")
lin, non, gap = (0.8, 0.05), (0.8, 0.05), 0.0
for t in range(HORIZON):
    lin = linear_step(*lin, u_ls[t], a_c[t], kappa_c[t])
    non = nonlinear_step(*non, u_ls[t], a_c[t], kappa_c[t])
    gap = max(gap, abs(lin[0] - non[0]))
print(f"along the least-squares plan, the two models' e differ by at most {gap:.3f} m")
assert exact.success
assert 0.0 <= above <= 0.02 and abs(above - 0.0094) <= 0.0005  # U10.1: within 2% (the probe: 0.94%)
assert np.max(np.abs(u_ls - exact.x)) <= 1e-3  # U10.1

The exact solver's J is 3.49971.
The least-squares plan, evaluated on the **nonlinear** model it never used, costs 3.53258: **0.94% more**.
Its commands differ from the exact ones by at most 3.5·10⁻⁴ 1/m, about 0.06° of wheel angle, and along the plan the two models stay within 0.1 m of each other: near the line, the linearisation costs under 1%.

The price of exactness is time.
`minimize` took hundreds of milliseconds (the plan's probe: 480 ms; it is printed above for this machine), several hundred times the least-squares solve, and far more than the 50 ms tick, let alone the 5 ms budget of §7.
That is why the submission solves the linear problem, with numpy only, and why `scipy` stays here, as the exact reference on one corner (a submission may not import it anyway).

## 5. Weights are tuned on the dynamic car, and the lag matters

The agent's weights are `MPC_Q_E = 0.3`, `MPC_Q_PSI = 20.0` and `MPC_R_DU = 1000.0`.
They were not the first weights tried.
The first were (q_e, q_ψ, r) = (1.0, 4.0, 400.0): a cheaper change cost and a heavier price on cross-track error.
Race them at Unit 9's 0.54 on `test-circuit`, on the carla-like car and on Unit 3's **kinematic** car (no noise, no slip), and race the shipped weights at the same 0.54 on the carla-like car:

In [ ]:
FIRST_WEIGHTS = {"MPC_Q_E": 1.0, "MPC_Q_PSI": 4.0, "MPC_R_DU": 400.0}
first_source = variant(UNIT_10, "first-weights-0.54", MU_FRACTION=0.54, **FIRST_WEIGHTS, **NO_TIMING)
first_dynamic = race(first_source)
first_kinematic = race(first_source, car=KinematicCar())
mpc_054 = race(variant(UNIT_10, "mpc-0.54", MU_FRACTION=0.54, **NO_TIMING))
print("first weights, the carla-like car:", summary(first_dynamic, line_offsets))
print("   ... its respawns:", ", ".join(sorted(set(first_dynamic.respawn_causes))))
print("first weights, the kinematic car: ", summary(first_kinematic, line_offsets))
print("shipped weights, the carla-like car:", summary(mpc_054, line_offsets))
assert not clean(first_dynamic, line_offsets)  # the first weights: not clean on the dynamic car (p1)
assert clean(first_kinematic, line_offsets) and abs(first_kinematic.total_time - 278.25) <= TICK
assert clean(mpc_054, line_offsets) and abs(mpc_054.total_time - 279.65) <= TICK  # U10.2

fig, ax = plt.subplots(figsize=(11, 3.5))
for name, result in (("first weights, carla-like car", first_dynamic), ("first weights, kinematic car", first_kinematic),
                     ("shipped weights, carla-like car", mpc_054)):
    telemetry = result.telemetry
    window = (telemetry["t"] > 20.0) & (telemetry["t"] < 60.0)
    target = line_offsets[(telemetry["progress"] + 1) % len(line_offsets)]
    error = np.where(telemetry["held"] == 0, telemetry["offset"] - target, np.nan)
    ax.plot(telemetry["t"][window], error[window], label=name)
ax.set_xlabel("t (s)")
ax.set_ylabel("error from the line (m, left +)")
ax.set_title("test-circuit, f = 0.54, seed 0: 20 to 60 s")
ax.legend()
plt.show()

On the kinematic car the first weights are excellent: clean in 278.25 s, never more than 0.29 m from the line.
On the carla-like car they are not clean: they spin 11 times and never finish.
The plot shows how it starts: on the fast, nearly straight stretch after 30 s (the line's radius there is over 100 m, at 30–37 m/s) the car starts to weave, about once a second; the swings grow, and at about 44 s, on a straight, it spins.
The model is kinematic and the car is not, and weights that are fine for the model can be wrong for the car.

So the shipped weights were tuned **on the carla-like car**, on the training track only (plan `roar-009`: 27 settings of (q_e, q_ψ, r), from {0.02, 0.1, 0.3} × {1, 4, 20} × {10³, 10⁴, 10⁵}, raced at 0.54 on `test-circuit`, seed 0; 12 were clean, two went on to the training family, and one survived it).
The winner pays much more for heading error (q_ψ = 20) and for changing the command (r = 1000), and less for cross-track error (q_e = 0.3): it corrects more gently.
On the carla-like car its error (green) is larger than the kinematic car's, but it stays under 1.04 m all race, and the swings do not grow.

**The lag.**
Now the shipped weights, but with `MPC_LAG = 0`: the model forgets the queued command and plans as if this tick's command acted at once.
Race it at 0.60 on `test-circuit`, where the agent with the lag is clean (§6):

In [ ]:
lag0 = race(variant(UNIT_10, "lag0-0.60", MU_FRACTION=0.60, MPC_LAG=0, **NO_TIMING))
print("MPC_LAG = 0, f = 0.60:", summary(lag0, line_offsets))
print("   ... its respawns:", ", ".join(sorted(set(lag0.respawn_causes))))
assert not clean(lag0, line_offsets)  # U10.6

Without the queued command in its model, the shipped MPC spins 17 times and never finishes (probe record, plan `roar-009` U10.6: not clean on any of seeds 0–4, while with the lag modelled it is clean on all five at 0.60, §6).
On the toy of §3, forgetting the lag cost nothing; on the carla-like car it is the difference between a clean race and no finish.
Like the weights, that is something only a race on the car could tell you.

## 6. The honest comparison

**At the same f.**
The hook compared pursuit and MPC at 0.58, where pursuit fails.
At Unit 9's own 0.54, where both are clean, which is faster?

In [ ]:
pursuit_054 = race(UNIT_09)  # Unit 9 as shipped: 0.54
print("Unit 9, pure pursuit, f = 0.54:", summary(pursuit_054, line_offsets))
print("Unit 10, MPC,          f = 0.54:", summary(mpc_054, line_offsets))
assert clean(pursuit_054, line_offsets) and abs(pursuit_054.total_time - 279.60) <= TICK

One tick apart: 279.60 s against 279.65 s.
Over seeds 0–4 (U10.2) **neither** is noise-aware better than the other, in either direction (Unit 1's rule).
That is the honest result: at a given f, the **speed profile** sets the lap time, not the steering, because both agents drive the same profile on the same line.
What MPC changes is how closely it follows the line: never more than 1.04 m from it, against pursuit's 1.52 m.
That **tracking margin** is MPC's gain.

**Re-tune by the margin rule.**
The tracking margin is worth something only if it is spent: on a higher fraction of the grip.
The probe record of plan `roar-009` (the five training tracks, seeds 0–4 at 0.60; "no" = not clean on at least one seed of a track; the 0.54–0.58 rows on seeds 0–1):

| f | pure pursuit (Unit 9) | MPC |
|---|---|---|
| 0.54 | clean, max error 1.74 m | clean, 1.20 m |
| 0.56 | clean, 1.88 m: **the edge** | clean, 1.22 m |
| 0.58 | no (`test-circuit-seed4`) | clean, 1.34 m |
| 0.60 | no | clean on all 25 races, 1.50 m: **the edge** |
| 0.62 | — | no (`test-circuit-seed1`: 2 of 5 clean) |

MPC's family edge is **0.60**, two grid steps above pursuit's 0.56.
The margin rule steps one grid step inside: `agents/unit-10.py` freezes **`MU_FRACTION = 0.58`**.
At 0.60 its medians are 272.40, 269.40, 268.80, 268.75 and 268.85 s on the five training tracks.

**The transfer.**
The fraction was tuned on the training family only; the test is the unseen `test-circuit-seed2026`:

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
transfer = race(on_track(UNIT_10, "test-circuit-seed2026", seed2026_control, "u10-seed2026", **NO_TIMING),
                track="test-circuit-seed2026")
print("Unit 10, MPC, f = 0.58, test-circuit-seed2026:", summary(transfer, seed2026_offsets))
assert clean(transfer, seed2026_offsets) and abs(transfer.total_time - 276.85) <= TICK  # U10.5

Clean in **276.85 s**, and over seeds 0–4 the median is 276.85 s (U10.5), against Unit 9's 282.05 s on the same track and seeds: a **noise-aware** win (Unit 1's rule; the spreads are at most 0.10 s), and the first one this unit has earned.
It comes from the fraction the margin bought, not from the steering being faster at a given fraction.

**`monza-like`.**
On the long track (probe record, seeds 0–4 at 0.64; the milestone re-races them) MPC is clean in 468.75 s, never more than 1.25 m from its line.
Pursuit at 0.64 also finishes every race in about 468.8 s with **no collision**, but it is 2.03–2.08 m from its line at worst, just over the clean rule's 2 m.
That is a **threshold miss**, not a crash: on this track MPC's gain is the tracking margin, measured, and nothing more.

## 7. The tick budget

An MPC agent solves a least-squares problem **every tick**, and the tick is 50 ms of the simulator's time.
`check_submission` asks for the 99th percentile (p99) of `step()` to be at most **5 ms**, with the first 50 steps (the warm-up: imports, caches, the first allocations) excluded.
Why p99 and not the average?
A step that is slow once in a hundred ticks is slow about once every 5 s of racing; the median says nothing about it.

`run_race` records how long every `step()` took (`step_ms`).
Read the hook's MPC race, raced by itself in this notebook:

In [ ]:
steps_ms = mpc_058.step_ms
print(f"{steps_ms.size} steps: p50 {np.percentile(steps_ms[50:], 50):.2f} ms, p99 {np.percentile(steps_ms[50:], 99):.2f} ms "
      f"after the warm-up; the slowest of the first 50: {np.max(steps_ms[:50]):.2f} ms")
print(f"pure pursuit's steps (the hook's Unit 9 race): p50 {np.percentile(pursuit_058.step_ms[50:], 50):.2f} ms")

# the solve alone, race-free, at four horizons
for m in (10, 20, 40, 60):
    v_m, k_m = horizon_reference(start, profile[start], m)
    times = []
    for _ in range(220):
        started = time.perf_counter()
        mpc_tick(0.8, 0.05, DT * v_m, k_m, [0.0], 0.0)
        times.append((time.perf_counter() - started) * 1000)
    print(f"horizon {m:2d}: p50 {np.percentile(times[20:], 50):.3f} ms, p99 {np.percentile(times[20:], 99):.3f} ms")

These numbers are this machine's, now; yours will differ, and the notebook asserts none of them.
The plan's probes measured p50 about 0.9 ms and p99 about **1.0 ms** for the whole step on an idle machine: about five times inside the budget.
Pure pursuit's step is about 0.16 ms.
The solve grows with the horizon, and the race time stops improving long before the cost does (plan `roar-009`, `test-circuit` at 0.60: N = 10, 273.05 s; 20, 272.45 s; 40, 272.40 s; 60, 272.40 s).

**Never time in a pool.**
The probes race eight races at once in a process pool to save time.
Timed that way, the same agent's p99 was **2 to 13 ms** in the plan's probes, sometimes over budget, because eight processes share the machine and any one of them can be paused for a few milliseconds.
A race time from a pool is fine (the simulator's clock does not care); a timing from a pool is not.
`check_submission` is always run alone, and so is every timing in this book.

**What a self-timed check can do.**
The agent cannot trust the machine it will race on, so it times its own `mpc()` call (the reference, the stack and the solve) with `time.perf_counter()`, against `MPC_BUDGET_MS = 5.0`.
It can only know a solve was slow **after** it has finished: that tick's time is already spent, and the slow solve's command is the best one it has, so it is applied.
What it can change is the future: the agent counts the tick in `over_budget` and **halves the horizon** for the rest of the race (40 → 20 → 10, never below `MPC_MIN_HORIZON = 10`), printing a line each time.
It never falls back to pure pursuit: the hook showed what pursuit does at MPC's fraction.
And if a plan is ever not finite (only a non-finite input, such as a bad sensor reading, can cause that), it is never applied; the agent applies the next command of its last finite plan instead and counts it in `fallbacks`.

**Why the pinned races switch the timing off.**
A race whose horizon depends on the wall clock depends on how busy the machine was: two runs of the same seed could shorten the horizon at different ticks and finish at different times.
So every race this book pins uses the variant `MPC_BUDGET_MS = None`, which never times.
The policy itself is raced at its other extreme, `MPC_BUDGET_MS = 0.0`: every solve is "over budget", so the horizon is 40 for one tick, 20 for one tick, and 10 for the rest of the race, deterministically:

In [ ]:
with shim_installed():
    Budget0, _ = load_submission(on_track(UNIT_10, "test-circuit-seed4", seed4_control, "u10-seed4-budget0",
                                          MPC_BUDGET_MS=0.0))
kept = {}


class Kept(Budget0):
    # the submission unchanged, except that it keeps a reference to itself for us to read after the race
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        kept["agent"] = self


budget0 = race(Kept, track="test-circuit-seed4")
budget0_agent = kept["agent"]
print("MPC_BUDGET_MS = 0.0:", summary(budget0, seed4_offsets))
print(f"steps {budget0.step_ms.size}, over_budget {budget0_agent.over_budget}, final horizon {budget0_agent.horizon}, "
      f"fallbacks {budget0_agent.fallbacks}")
assert clean(budget0, seed4_offsets) and abs(budget0.total_time - 271.60) <= TICK  # U10.7 (median 271.55 s)
assert budget0_agent.over_budget == budget0.step_ms.size and budget0_agent.horizon == 10  # U10.7
assert budget0_agent.fallbacks == 0  # U10.7

The two printed lines are the agent shortening its horizon, on its first two ticks.
At N = 10 for the whole race it is still clean, in 271.60 s against 271.10 s at N = 40 (seeds 0–4: 271.55 s, all clean, U10.7; on `test-circuit-seed2026` 277.30 s, clean, against 276.85 s): a shorter horizon costs about half a second, not the race.
That is the evidence the policy rests on.
The first idea, handing a slow tick to pure pursuit, would break the change cost and the lag queue, and pure pursuit throughout at 0.58 is what crashed in the hook.

## 8. To the real car: the lab

Everything here was measured on racekit's carla-like car, which is a model too.
This unit's lab, `labs/real-sim.ipynb` (it runs on your machine with CARLA 0.9.12 and `ROAR_PY`), carries MPC to CARLA, carefully:

- it starts from the **accepted configuration**, the exact submission your Unit 8 lab accepted after clean CARLA races; nothing un-raced is ever the fallback;
- **timing first, off CARLA**: `check_submission` on your MPC file, on the machine you will race on, idle; if it fails, try `MPC_HORIZON = 20`; if that fails too, you keep the accepted configuration;
- then two races of the MPC file with the accepted configuration's constants; both clean, and MPC is accepted at that fraction; otherwise you keep the accepted configuration;
- only then, the fraction plus 0.02, twice; the lab never ends on a value it did not race.

**What this unit adds to the toolbox:**

- a model of the **error**, not of the world: two numbers in the path frame, linearised about where you want to be;
- a **time-varying** linear model, read off your own plan, with the actuator lag rolled in;
- a horizon cost that **stacks** into one least-squares problem, built in closed form, solved once a tick;
- an **exact reference** on one case, to know what the shortcut costs (here, under 1%);
- weights tuned on the car that is **not** the model, and the honest comparison: same speed at the same f, a tracking margin that buys a higher f;
- and a compute budget measured alone, with a policy for when it is missed.

Next unit: robustness, "one crash costs you a lap", where the agent learns to notice when things go wrong and to survive cars that are not quite the one it was tuned on.